In [37]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
import numpy as np
from src.params import *

from src.preprocess import *

import torch
from torch import nn
from torch.utils.data import Dataset, Subset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from pytorch_lightning import LightningDataModule, LightningModule

In [2]:
df = pd.read_csv(".." /DATA_DIR / "train.csv")

In [3]:
df

,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106795,351917269,6,12.0,2147388374,6,12.0,4195677307,10351,LRDA,0,0,0,3,0,0
106796,351917269,7,14.0,2147388374,7,14.0,290896675,10351,LRDA,0,0,0,3,0,0
106797,351917269,8,16.0,2147388374,8,16.0,461435451,10351,LRDA,0,0,0,3,0,0
106798,351917269,9,18.0,2147388374,9,18.0,3786213131,10351,LRDA,0,0,0,3,0,0


In [4]:
mean_test = 0.237168
std_test =  0.222340

# Construction custom dataset

In [ ]:
class BrainDataset(Dataset):

    def __init__(self, metadata, mean, std):

        super().__init__()
        self.metadata = metadata #metadata is the train.csv file
        self.mean = mean #mean and std stream-calculated in the datamodule over the train set
        self.std = std

    def len(self):
        return len(self.metadata)

    def __getitem__(self, idx):

        #get file name
        spec_id = self.metadata.iloc[idx]["spectrogram_id"]
        subsample_id = self.metadata.iloc[idx]["spectrogram_sub_id"]

        #load file
        path = PROCESSED_DIR / f"{spec_id}-{subsample_id}.npy"
        spec = np.load(path)

        #log transfo
        spec = np.log1p(spec) #should be before norm?

        #normalization
        spec = (spec - self.mean)/self.std # add epsilon to avoid break if self.std = 0?

        #conversion en tensor
        spec = torch.tensor(spec, dtype= torch.float32)

        #get votes
        votes = self.metadata.iloc[idx][VOTE_COL].values.astype(int)
        #print(votes)
        #print(votes.dtype)
        #convert into dist
        votes = votes/votes.sum()
        #convert into tensor
        votes = torch.from_numpy(votes).float()
        #print(votes.dtype)


        return spec, votes

In [29]:
test_dataset = BrainDataset(metadata= df, mean= mean_test, std= std_test)


In [35]:
test_dataset.__getitem__(idx= 0)[1]

[3 0 0 0 0 0]
int64
torch.float32


tensor([1., 0., 0., 0., 0., 0.])

In [ ]:
class BrainDataModule(LightningDataModule):

    def __init__(self, metadata, dataset, batch_size= 32, num_workers= 4, seed= 273, n_split= 5, n_fold = 0):

        super().__init__()
        self.dataset = dataset #BrainDataset
        self.metadata = metadata # train.csv file, need again for split.
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.seed = seed
        self.n_split = max(5, n_split)
        self.n_fold = max(0, n_fold)


    def setup(self, stage= None):

        #split
        sgkf = StratifiedGroupKFold(n_splits= self.nb_fold)
        groups = self.metadata["patient_id"]
        X = range(len(self.metadata))
        y = self.metadata["expert_consensus"]
        sgkf.get_n_splits()

        for i, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
            if i < self.nb_fold:
                continue

            if i == self.nb_fold:

                self.train_idx = train_idx
                self.val_idx = val_idx
                break

        if stage in ("fit", None):

            self.train_ds = Subset(self.dataset, self.train_idx)
            self.val_ds = Subset(self.dataset, self.val_idx)

    def train_dataloader(self):
        return DataLoader(self.train_ds,
                          batch_size= self.batch_size,
                          shuffle= True,
                          num_workers= self.num_workers,
                          pin_memory= True,
                          persistent_workers= self.num_workers > 0)


    def val_dataloader(self):
        return DataLoader(self.val_ds,
                          batch_size= self.batch_size,
                          shuffle= False,
                          num_workers= self.num_workers,
                          pin_memory= True,
                          persistent_workers= self.num_workers > 0)


In [ ]:
#how many annotation per patient
df["patient_id"].value_counts(normalize= True)*100
# => big imbalance between patients.

patient_id
30631    2.073970
2641     2.045880
35627    1.313670
28330    1.275281
54199    1.264045
           ...   
4539     0.000936
41339    0.000936
43246    0.000936
33103    0.000936
38896    0.000936
Name: proportion, Length: 1950, dtype: float64